## HDC(Cosine Similarity)

In [32]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sobol_seq import i4_sobol_generate

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

# Load and shuffle dataset
file_path = 'heart.csv'
X, y = load_dataset(file_path)
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2000  # Experiment with different values
NUM_CLASSES = len(np.unique(y))
NUM_SAMPLES = X_train.shape[0]

# Create Sobol sequence for random projection matrix
print("Generating Sobol sequence...")
sobol_sequence = i4_sobol_generate(X_train.shape[1], D)

def project_data(data, seq):
    return np.dot(data, seq.T)

def classify(images, class_hypervectors):
    similarities = cosine_similarity(images, class_hypervectors)
    classifications = np.argmax(similarities, axis=1)
    return classifications

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, sobol_sequence)

# Create class hypervectors by summing the projected vectors of each class
class_hypervectors = np.zeros((NUM_CLASSES, D))
for i in range(NUM_SAMPLES):
    class_hypervectors[y_train[i]] += X_train_proj[i]

# Normalize the class hypervectors
class_hypervectors = class_hypervectors / np.linalg.norm(class_hypervectors, axis=1, keepdims=True)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_proj, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100
print(f"Conventional HDC (Cosine Similarity) Train Accuracy: {acc_train:.2f}%")

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, sobol_sequence)

# Classify test data
predictions_test = classify(X_test_proj, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100
print(f"Conventional HDC (Cosine Similarity) Test Accuracy: {acc_test:.2f}%")

# Calculate model memory requirement
model_memory = sobol_sequence.nbytes + class_hypervectors.nbytes

print(f"Conventional HDC (Cosine Similarity) Training Time: {training_time:.4f} seconds")
print(f"Conventional HDC (Cosine Similarity) Inference Time: {inference_time:.4f} seconds")
print(f"Conventional HDC (Cosine Similarity) Model Memory Required: {model_memory / (1024 * 1024):.4f} MB")

Generating Sobol sequence...
Conventional HDC (Cosine Similarity) Train Accuracy: 79.30%
Conventional HDC (Cosine Similarity) Test Accuracy: 78.76%
Conventional HDC (Cosine Similarity) Training Time: 0.0050 seconds
Conventional HDC (Cosine Similarity) Inference Time: 0.0000 seconds
Conventional HDC (Cosine Similarity) Model Memory Required: 0.2289 MB


## HDC(Hamming Distance)

In [16]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sobol_seq import i4_sobol_generate

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

# Load and shuffle dataset
file_path = 'heart.csv'
X, y = load_dataset(file_path)
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2000  # Experiment with different values
NUM_CLASSES = len(np.unique(y))
NUM_SAMPLES = X_train.shape[0]

# Create Sobol sequence for random projection matrix
print("Generating Sobol sequence...")
sobol_sequence = i4_sobol_generate(X_train.shape[1], D)

def project_data(data, seq):
    return np.dot(data, seq.T)
    
def binarize_data(data, threshold):
    """Binarize data by thresholding"""
    return (data > threshold).astype(int)

def hamming_distance(x, y):
    """Compute Hamming distance between two binary vectors"""
    return np.sum(x != y, axis=1)

def classify(images, class_hypervectors):
    distances = np.zeros((images.shape[0], class_hypervectors.shape[0]))
    for i in range(class_hypervectors.shape[0]):
        distances[:, i] = hamming_distance(images, class_hypervectors[i])
    classifications = np.argmin(distances, axis=1)
    return classifications

def evaluate_thresholds(X_train, X_test, y_train, y_test):
    best_threshold = 0
    best_accuracy = 0
    for threshold in np.linspace(0, 1, 21):  # Try thresholds from 0 to 1
        X_train_bin = binarize_data(X_train, threshold)
        X_test_bin = binarize_data(X_test, threshold)

        # Compute class hypervectors
        class_hypervectors = np.zeros((2, X_train_bin.shape[1]), dtype=int)
        for i in range(X_train_bin.shape[0]):
            class_hypervectors[y_train[i]] += X_train_bin[i]

        # Binarize the class hypervectors
        class_hypervectors = binarize_data(class_hypervectors, threshold)
        predictions_test = classify(X_test_bin, class_hypervectors)
        acc_test = accuracy_score(y_test, predictions_test) * 100
        
        if acc_test > best_accuracy:
            best_accuracy = acc_test
            best_threshold = threshold
    
    return best_threshold, best_accuracy

# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Measure training time
start_train_time = time.time()

# Optimize threshold
best_threshold, best_accuracy = evaluate_thresholds(X_train, X_test, y_train, y_test)

# Binarize training data with best threshold
X_train_bin = binarize_data(X_train, best_threshold)

# Create class hypervectors by summing the binarized vectors of each class
class_hypervectors = np.zeros((2, X_train_bin.shape[1]), dtype=int)
for i in range(X_train_bin.shape[0]):
    class_hypervectors[y_train[i]] += X_train_bin[i]

# Binarize the class hypervectors with the best threshold
class_hypervectors = binarize_data(class_hypervectors, best_threshold)

# Measure training time
end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = classify(X_train_bin, class_hypervectors)
acc_train = accuracy_score(y_train, predictions_train) * 100

# Measure inference time
start_inference_time = time.time()

# Binarize test data with best threshold
X_test_bin = binarize_data(X_test, best_threshold)

# Classify test data
predictions_test = classify(X_test_bin, class_hypervectors)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

# Test accuracy
acc_test = accuracy_score(y_test, predictions_test) * 100

# Calculate model memory requirement
model_memory = class_hypervectors.nbytes

print(f"Best Threshold: {best_threshold}")
print(f"Conventional HDC (Hamming Distance) Train Accuracy: {acc_train:.2f}%")
print(f"Conventional HDC (Hamming Distance) Test Accuracy: {acc_test:.2f}%")
print(f"Conventional HDC (Hamming Distance) Training Time: {training_time:.4f} seconds")
print(f"Conventional HDC (Hamming Distance) Inference Time: {inference_time:.4f} seconds")
print(f"Conventional HDC (Hamming Distance) Model Memory Required: {model_memory / (1024 * 1024):.4f} MB")

Generating Sobol sequence...
Best Threshold: 0.0
Conventional HDC (Hamming Distance) Train Accuracy: 48.69%
Conventional HDC (Hamming Distance) Test Accuracy: 48.67%
Conventional HDC (Hamming Distance) Training Time: 0.0212 seconds
Conventional HDC (Hamming Distance) Inference Time: 0.0000 seconds
Conventional HDC (Hamming Distance) Model Memory Required: 0.0001 MB


## SVM

In [18]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sobol_seq import i4_sobol_generate

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

# Load and shuffle dataset
file_path = 'heart.csv'
X, y = load_dataset(file_path)
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2000  # Experiment with different values
NUM_CLASSES = len(np.unique(y))
NUM_SAMPLES = X_train.shape[0]

# Create Sobol sequence for random projection matrix
print("Generating Sobol sequence...")
sobol_sequence = i4_sobol_generate(X_train.shape[1], D)

def project_data(data, seq):
    return np.dot(data, seq.T)
# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, sobol_sequence)

# Train an SVM classifier on the projected data
svm_classifier = SVC(kernel='linear')
svm_classifier.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = svm_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train) * 100
print("Convensional SVM Train Accuracy: ", acc_train)

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, sobol_sequence)

# Classify test data using the trained SVM classifier
predictions_test = svm_classifier.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print("Convensional SVM Test Accuracy: ", acc_test)

# Calculate model memory requirement
model_memory = sobol_sequence.nbytes + sys.getsizeof(svm_classifier)

print("Convensional SVM Training Time: {:.4f} seconds".format(training_time))
print("Convensional SVM Inference Time: {:.4f} seconds".format(inference_time))
print("Convensional SVM Model Memory: {:.4f} MB".format(model_memory / (1024 * 1024)))

Generating Sobol sequence...
Convensional SVM Train Accuracy:  84.40233236151603
Convensional SVM Test Accuracy:  82.89085545722715
Convensional SVM Training Time: 0.4782 seconds
Convensional SVM Inference Time: 0.0473 seconds
Convensional SVM Model Memory: 0.1984 MB


## MLP

In [20]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sobol_seq import i4_sobol_generate

def shuffle(X, y):
    permutation = np.arange(X.shape[0])
    np.random.shuffle(permutation)
    return X[permutation], y[permutation]

def load_dataset(file_path):
    df = pd.read_csv(file_path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y

# Load and shuffle dataset
file_path = 'heart.csv'
X, y = load_dataset(file_path)
X, y = shuffle(X, y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Normalize the data using MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Set dimensions for hyperdimensional space
D = 2000  # Experiment with different values
NUM_CLASSES = len(np.unique(y))
NUM_SAMPLES = X_train.shape[0]

# Create Sobol sequence for random projection matrix
print("Generating Sobol sequence...")
sobol_sequence = i4_sobol_generate(X_train.shape[1], D)

def project_data(data, seq):
    return np.dot(data, seq.T)
    
# Normalize data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Measure training time
start_train_time = time.time()

# Project training data to hyperdimensional space
X_train_proj = project_data(X_train, sobol_sequence)

# Train an MLP classifier on the projected data
mlp_classifier = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)  # Adjust hyperparameters
mlp_classifier.fit(X_train_proj, y_train)

end_train_time = time.time()
training_time = end_train_time - start_train_time

# Training accuracy
predictions_train = mlp_classifier.predict(X_train_proj)
acc_train = accuracy_score(y_train, predictions_train) * 100
print(f"Conventional MLP Train Accuracy: {acc_train:.2f}%")

# Measure inference time
start_inference_time = time.time()

# Project test data to hyperdimensional space
X_test_proj = project_data(X_test, sobol_sequence)

# Classify test data using the trained MLP classifier
predictions_test = mlp_classifier.predict(X_test_proj)

end_inference_time = time.time()
inference_time = end_inference_time - start_inference_time

acc_test = accuracy_score(y_test, predictions_test) * 100
print(f"Conventional MLP Test Accuracy: {acc_test:.2f}%")

# Calculate model memory requirement
model_memory = sobol_sequence.nbytes + sys.getsizeof(mlp_classifier)

print(f"Conventional MLP Training Time: {training_time:.4f} seconds")
print(f"Conventional MLP Inference Time: {inference_time:.4f} seconds")
print(f"Conventional MLP Model Memory: {model_memory / (1024 * 1024):.4f} MB")

Generating Sobol sequence...
Conventional MLP Train Accuracy: 100.00%
Conventional MLP Test Accuracy: 97.35%
Conventional MLP Training Time: 5.6831 seconds
Conventional MLP Inference Time: 0.0040 seconds
Conventional MLP Model Memory: 0.1984 MB
